# Unit 6 Exercise  

**Zen Anthony Pastolero BSCS 3-A AI** 


---
## Part 1: Fine-Tuning a Pretrained LLM

Using the available pretrained LLMs on HuggingFace, a dataset made for a specific domain is selected and used for fine-tuning.

---

### Step 1: Identified the Task to be Performed

**Task:** Sequence Classification — specifically, **text classification** into predefined categories.

Sequence classification is the task of assigning a label or class to a given input text sequence. In this exercise, the model is trained to read a news article and predict which of four categories it belongs to.

---

### Step 2: Identified the Domain Used for Fine-Tuning

**Domain:** General News

**Dataset:** [`ag_news`](https://huggingface.co/datasets/ag_news) from Hugging Face.

The AG News dataset is a collection of news articles gathered from more than 2,000 news sources. Each article is classified into one of four categories:

| Label | Category  |
|-------|-----------|
| 0     | World     |
| 1     | Sports    |
| 2     | Business  |
| 3     | Sci/Tech  |

---

### Step 3: Identified the LLM to be Used

**Model:** `distilbert-base-uncased`

`distilbert-base-uncased` is a smaller, faster, and lighter version of the BERT model, retaining approximately 97% of BERT's language understanding capabilities while being 40% smaller and 60% faster. This makes it ideal for fine-tuning on limited hardware (such as a standard laptop or Colab environment) while still achieving strong classification performance.

---

### Step 4: Established the Configuration Needed for Fine-Tuning

The fine-tuning pipeline uses the **Hugging Face `Trainer` API** with **PyTorch** as the backend.

**Training Configuration:**

| Hyperparameter                  | Value         |
|---------------------------------|---------------|
| Learning Rate                   | `2e-5`        |
| Per-Device Train Batch Size     | `8`           |
| Per-Device Eval Batch Size      | `8`           |
| Number of Epochs                | `3`           |
| Training Subset Size            | `1000` samples|
| Evaluation Subset Size          | `500` samples |
| Evaluation Strategy             | Per epoch     |

A smaller subset of the full AG News dataset is used to ensure the notebook executes efficiently within resource constraints.

---

### Step 5: Performed Evaluation

The model is evaluated using the **`accuracy` metric** from the Hugging Face `evaluate` library. Validation accuracy and validation loss are computed after each training epoch and printed upon completion of training.


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# 1. Load Dataset and Tokenizer
# ─────────────────────────────────────────────────────────────────────
# Load the AG News dataset from Hugging Face Hub.
# The dataset has 'train' and 'test' splits with 'text' and 'label' columns.
dataset = load_dataset("ag_news")
print("Dataset loaded successfully.")
print(dataset)

# Load the DistilBERT tokenizer.
# 'distilbert-base-uncased' converts all text to lowercase before tokenizing.
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print("\nTokenizer loaded successfully.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# 2. Tokenization Strategy
# ─────────────────────────────────────────────────────────────────────
# Define a tokenization function that:
#   - Applies padding to make all sequences the same length ('max_length')
#   - Applies truncation to cut sequences longer than the model's max input
# This is applied to the entire dataset using .map() for efficiency.
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Tokenization complete.")
print(tokenized_datasets)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Use a small subset for faster training in this exercise
# ─────────────────────────────────────────────────────────────────────
# The full AG News dataset is large (~120,000 train samples).
# We select 1,000 training samples and 500 evaluation samples with a fixed
# random seed (42) for reproducibility.
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset  = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

print(f"Training samples  : {len(small_train_dataset)}")
print(f"Evaluation samples: {len(small_eval_dataset)}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# 3. Initialize Model
# ─────────────────────────────────────────────────────────────────────
# Load DistilBERT with a sequence classification head on top.
# num_labels=4 because AG News has 4 classes: World, Sports, Business, Sci/Tech.
# The model weights from the pretrained checkpoint are loaded, and the
# classification head is randomly initialized (it will be trained from scratch).
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)
print("Model initialized successfully.")
print(f"Number of parameters: {model.num_parameters():,}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# 4. Setup Training Arguments
# ─────────────────────────────────────────────────────────────────────
# TrainingArguments controls all aspects of training:
#   - output_dir        : where checkpoints are saved
#   - evaluation_strategy : evaluate after each epoch to monitor overfitting
#   - learning_rate     : 2e-5 is standard for fine-tuning BERT-family models
#   - batch sizes       : 8 per device (safe for most GPUs/CPUs)
#   - num_train_epochs  : 3 epochs is sufficient for fine-tuning on this dataset
training_args = TrainingArguments(
    output_dir="model_output",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
)

print("Training arguments configured:")
print(f"  Learning Rate      : {training_args.learning_rate}")
print(f"  Train Batch Size   : {training_args.per_device_train_batch_size}")
print(f"  Eval Batch Size    : {training_args.per_device_eval_batch_size}")
print(f"  Epochs             : {training_args.num_train_epochs}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# 5. Setup Evaluation Metric
# ─────────────────────────────────────────────────────────────────────
# We use the 'accuracy' metric from the Hugging Face evaluate library.
# compute_metrics is called after each evaluation pass:
#   - logits: raw model output scores for each class
#   - labels: the true class labels
#   - argmax selects the class with the highest score as the prediction
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

print("Evaluation metric (accuracy) loaded successfully.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# 6. Initialize Trainer
# ─────────────────────────────────────────────────────────────────────
# The Trainer handles the full training loop, gradient updates,
# evaluation, checkpointing, and logging automatically.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer initialized successfully.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# 7. Execute Training and Evaluation
# ─────────────────────────────────────────────────────────────────────
print("Starting Fine-Tuning...")
print("=" * 60)

trainer.train()

print("\n" + "=" * 60)
print("Fine-Tuning Complete!")
print("=" * 60)
print("\nEvaluating Model on Validation Set...")

evaluation_results = trainer.evaluate()

print("\n--- Evaluation Results ---")
for key, value in evaluation_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

---
## Part 2: PCA Visualization of Word Vectors

**(15 points)** Visualize the vectors using PCA. Generate at least 20 known words and feed it on the visualization.

**Approach:**
- Load pre-trained **GloVe** word vectors (`glove-wiki-gigaword-50`) via the `gensim` library.
- GloVe represents each word as a 50-dimensional vector encoding semantic meaning.
- **Principal Component Analysis (PCA)** is applied to reduce 50 dimensions → 2 dimensions for 2D visualization.
- The resulting plot shows how semantically related words cluster together in vector space.


In [ ]:
import gensim.downloader as api
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# ─────────────────────────────────────────────────────────────────────
# Load Pre-Trained Word Vectors
# ─────────────────────────────────────────────────────────────────────
# 'glove-wiki-gigaword-50' provides 50-dimensional GloVe embeddings
# trained on Wikipedia + Gigaword text corpora (~400K vocabulary).
# The file is ~66MB and will be downloaded/cached on first run.
print("Downloading/Loading Word Vectors (GloVe 50-dim)...")
word_model = api.load("glove-wiki-gigaword-50")
print("Word vectors loaded successfully!")
print(f"Vocabulary size : {len(word_model.key_to_index):,} words")
print(f"Vector dimension: {word_model.vector_size}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# 20 Known Words for PCA Visualization
# ─────────────────────────────────────────────────────────────────────
# These 20 words span diverse semantic categories:
# nature (river, mountain, ocean, forest, sunlight),
# technology (computer, telephone, galaxy, planet),
# everyday objects (apple, guitar, blanket, mirror, window, coffee),
# abstract concepts (happiness, freedom, adventure, silence)
words = [
    "apple",     "river",     "computer",  "happiness", "mountain",
    "building",  "guitar",    "freedom",   "ocean",     "planet",
    "window",    "telephone", "galaxy",    "sunlight",  "coffee",
    "blanket",   "mirror",    "adventure", "silence",   "forest"
]

print(f"Selected {len(words)} words for visualization:")
print(words)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Extract Word Vectors and Apply PCA
# ─────────────────────────────────────────────────────────────────────
# Retrieve the 50-dimensional GloVe vector for each word.
# Stack them into a 2D array of shape (20, 50).
vectors = np.array([word_model[word] for word in words])
print(f"Word vector matrix shape: {vectors.shape}  (words × dimensions)")

# Apply PCA to reduce from 50D → 2D.
# PCA finds the two directions of maximum variance in the embedding space,
# allowing us to visualize semantic relationships on a 2D plane.
pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(vectors)

print(f"\nPCA explained variance ratio:")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.4f} ({pca.explained_variance_ratio_[0]*100:.2f}%)")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.4f} ({pca.explained_variance_ratio_[1]*100:.2f}%)")
print(f"  Total explained: {sum(pca.explained_variance_ratio_)*100:.2f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Plot the PCA Visualization
# ─────────────────────────────────────────────────────────────────────
plt.figure(figsize=(12, 8))

# Scatter plot: each point represents one word in 2D PCA space
plt.scatter(
    vectors_2d[:, 0],
    vectors_2d[:, 1],
    edgecolors='k',
    c='skyblue',
    s=100,
    zorder=3
)

# Annotate each point with the corresponding word
for word, (x, y) in zip(words, vectors_2d):
    plt.annotate(
        word,
        (x, y),
        textcoords="offset points",
        xytext=(0, 10),
        ha='center',
        fontsize=10
    )

# Labels, title, and formatting
plt.title('PCA Visualization of 20 Random Word Vectors (GloVe 50-dim)', fontsize=14, fontweight='bold')
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('pca_word_vectors.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as 'pca_word_vectors.png'")